In [6]:
import pandas as pd
from pycaret.classification import setup, compare_models, tune_model, pull
from sklearn.exceptions import NotFittedError

# Load your raw dataset
df_original = pd.read_csv("data.csv")

# ---- Feature groups (for convenience) ----
preop_basic = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension"
]

intraop_basic = [
    "Surgical Approach",                 # raw categorical in df_original
    "Solid, cystic, mixed",              # raw categorical in df_original
    "Resection outcome (total, subtotal, partial)"
]

location_histology = [
    "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)",
    "Tumor (Adamantinomatous, papillary)",
    "BRAF+"
]

# ---- Build 15 experiments (targets + feature sets) ----
experiments = [
    # Post-op complications
    {"target": "Permanent DI",
     "features": preop_basic + ["Surgical Approach", "Resection outcome (total, subtotal, partial)"]},
    {"target": "Recurrence after this operation",
     "features": preop_basic + ["Surgical Approach", "Resection outcome (total, subtotal, partial)"]}

    # Composite endpoints (you can create these columns beforehand if desired)
    # Example creation idea (run before the loop):
    # df_original["Any complication"] = (
    #     (df_original["CSF leak"]==1) | (df_original["Transient DI"]==1) |
    #     (df_original["Permanent DI"]==1) | (df_original["Other vascular complication (need to specify)"]==1) |
    #     (df_original["Hemorrhage or hemorrhagic stroke"]==1) | (df_original["Ischemic Stroke"]==1) |
    #     (df_original["Other intracranial infection"]==1) | (df_original["Non-intracranial infection"]==1) |
    #     (df_original["Meningitis"]==1)
    # ).astype(int)

    # If you already created it, uncomment the experiment below:
    # {"target": "Any complication",
    #  "features": preop_basic + intraop_basic + location_histology},

    # Visual outcome (binary version you can precompute; multi-class needs classification setup too)
    # df_original["VisualImproved"] = (df_original[
    #   "Post-op visual improvement/stable/worse (subjective); should add something specific about visual fields if any patients have; add about visual acuity"
    # ].astype(str).str.lower().str.contains("improv")).astype(int)

    # {"target": "VisualImproved",
    #  "features": preop_basic + ["Surgical Approach"] + location_histology},

    # Two more variations to reach 15 now (without needing new columns):

]

# ---- Utility: safe compare + tune ----
def run_experiment(df, target, features, metric="AUC"):
    # Keep only columns that exist (avoid KeyError if some columns are missing)
    cols = [c for c in features + [target] if c in df.columns]
    if target not in cols:
        print(f"  [skip] Target column '{target}' not found.")
        return None, None

    # Drop rows where target is missing
    data = df[cols].copy()
    data = data[~data[target].isna()]
    if data[target].nunique() < 2:
        print(f"  [skip] Target '{target}' has <2 classes after dropping NaNs.")
        return None, None

    # PyCaret setup
    clf_setup = setup(
        data=data,
        target=target,
        session_id=42,
        fold=3,                      # safer with rare events
        fold_strategy="stratifiedkfold",
        fix_imbalance=True,          # handle class imbalance
        use_gpu=False,
        verbose=False
    )

    # Compare and enforce single estimator
    best_model = compare_models(sort=metric, n_select=1)
    # Guard: sometimes returns list
    if isinstance(best_model, list):
        best_model = best_model[0]

    if not hasattr(best_model, "fit"):
        print("  [skip] No valid estimator returned by compare_models.")
        return None, None

    # Tune
    try:
        tuned = tune_model(best_model, optimize=metric)
    except Exception as e:
        print(f"  [warn] tune_model failed for '{target}': {e}")
        tuned = best_model

    leaderboard = pull()
    return tuned, leaderboard

# ---- Run all experiments ----
all_leaderboards = {}
for exp in experiments:
    print(f"\n=== Running target: {exp['target']} ===")
    try:
        tuned, leaderboard = run_experiment(df_original, exp["target"], exp["features"], metric="AUC")
        if leaderboard is not None:
            all_leaderboards[exp["target"]] = leaderboard.copy()
            # Show the top few rows for quick inspection
            print(leaderboard.head())
    except Exception as e:
        print(f"  [error] {exp['target']}: {e}")

# Optionally, concatenate and save all results
if all_leaderboards:
    lb_all = pd.concat({k: v for k, v in all_leaderboards.items()}, names=["Target", "Row"]).reset_index(level=0)
    lb_all.to_csv("automl_leaderboards.csv", index=False)
    print("\nSaved: automl_leaderboards.csv")


ModuleNotFoundError: No module named 'pycaret'